# 04 · PDFs & Excel Spreadsheets

**Module 14 – Working with Data & the Web** · Estimated time: 45 min

> Based on 📗 *Working with PDFs* (pypdf) and *PDFs & Spreadsheets Puzzle*. ⚠️ Fixes a bug in the original text-extraction loop, and ❗ adds **Excel** (`openpyxl`), which the section title promises but never covers. (CSV is covered in Module 11 · 03.)

### 🎯 What you will learn
- Reading PDFs and extracting text with `pypdf`
- Writing, merging, splitting, rotating and encrypting PDFs
- Creating new PDFs (overview of `fpdf2` / ReportLab)
- Reading and writing Excel `.xlsx` files with `openpyxl`: cells, rows, formulas, styling
- The 📗 puzzle: a hidden link in a CSV and a phone number in a PDF

## Part 1: PDFs with `pypdf`

> ⚠️ **Library history:** older tutorials use **`PyPDF2`** with camelCase methods (`PdfFileReader`, `getPage()`, `numPages`, `extractText()`). PyPDF2 is **deprecated**; its development merged back into **`pypdf`** (`pip install pypdf`), with `PdfReader`, `reader.pages[i]`, `len(reader.pages)`, `extract_text()`. The bootcamp has already been updated to `pypdf`, which is what we use here.

> ⚠️ Not every PDF has extractable text. Scanned PDFs are images and need OCR (e.g. `pytesseract`), and complex layouts may extract in an odd order.

In [1]:
import pypdf
print(pypdf.__version__)

reader = pypdf.PdfReader("Working_Business_Proposal.pdf")    # a path works; no need to open in "rb" manually
print("Pages:", len(reader.pages))
print("Metadata:", reader.metadata)

page_one = reader.pages[0]
page_one_text = page_one.extract_text()
print(page_one_text[:400])

6.21.1
Pages: 5
Metadata: {'/Title': IndirectObject(41, 0, 4574053776), '/Producer': IndirectObject(42, 0, 4574053776), '/Creator': IndirectObject(43, 0, 4574053776), '/CreationDate': IndirectObject(44, 0, 4574053776), '/ModDate': IndirectObject(44, 0, 4574053776)}
Business Proposal The Revolution is Coming 
Leverage agile frameworks to provide a robust synopsis for high level 
overviews. Iterative approaches to corporate strategy foster collaborative 
thinking to further the overall value proposition. Organically grow the 
holistic world view of disruptive innovation via workplace diversity and 
empowerment. 
Bring to the table win-win survival strategies t


### Extracting the text of every page

> ⚠️ **Bug in the original bootcamp notebook:** its loop was
> ```python
> for p in range(len(pdf_reader.pages)):
>     page = pdf_reader.pages[0]          # ❌ always page 0!
> ```
> so every entry contained page 1's text. Iterate over the pages directly:

In [2]:
pdf_text = [page.extract_text() for page in reader.pages]     # ✅ one entry per page
print(len(pdf_text))
print(pdf_text[3][:300])

5
competencies. Dramatically mesh low-risk high-yield alignments before 
transparent e-tailers. 
Appropriately empower dynamic leadership skills after business portals. 
Globally myocardinate interactive supply chains with distinctive quality 
vectors. Globally revolutionize global sources through int


### Writing / copying pages (📗)

You can't edit the text of an existing PDF easily, but you can copy, combine, rotate and reorder pages:

In [3]:
writer = pypdf.PdfWriter()
writer.add_page(reader.pages[0])
with open("Some_New_Doc.pdf", "wb") as f:
    writer.write(f)
print(len(pypdf.PdfReader("Some_New_Doc.pdf").pages), "page written")

1 page written


### Merging, splitting, rotating, encrypting

In [4]:
# Merge: append whole documents
merger = pypdf.PdfWriter()
merger.append("Working_Business_Proposal.pdf")
merger.append("Some_New_Doc.pdf")
merger.write("merged.pdf")
merger.close()
print("merged pages:", len(pypdf.PdfReader("merged.pdf").pages))

# Split: one file per page
for i, page in enumerate(reader.pages, start=1):
    w = pypdf.PdfWriter()
    w.add_page(page)
    w.write(f"page_{i}.pdf")
print("split into", len(reader.pages), "files")

# Rotate + encrypt
w = pypdf.PdfWriter()
for page in reader.pages[:2]:
    w.add_page(page).rotate(90)
w.encrypt(user_password="secret123")
w.write("rotated_locked.pdf")
locked = pypdf.PdfReader("rotated_locked.pdf")
print("encrypted:", locked.is_encrypted, "| decrypt:", locked.decrypt("secret123"))

merged pages: 6
split into 5 files
encrypted: True | decrypt: 2


### Creating brand-new PDFs
`pypdf` manipulates existing PDFs. To **generate** PDFs with text, tables and images, use:
- **`fpdf2`**: simple (`pip install fpdf2`)
- **ReportLab**: powerful, used for invoices and reports
- Or render HTML to PDF with **WeasyPrint**

```python
from fpdf import FPDF
pdf = FPDF()
pdf.add_page()
pdf.set_font("Helvetica", size=16)
pdf.cell(text="Invoice #1042", new_x="LMARGIN", new_y="NEXT")
pdf.output("invoice.pdf")
```

## Part 2: Excel files with `openpyxl`

`pip install openpyxl`. Works with `.xlsx` (not the old `.xls`).

In [5]:
from openpyxl import Workbook, load_workbook
from openpyxl.styles import Font, PatternFill, Alignment
from openpyxl.utils import get_column_letter

wb = Workbook()
ws = wb.active
ws.title = "Sales"

ws.append(["Region", "Q1", "Q2", "Q3", "Q4", "Total"])          # header row
data = [("North", 120, 135, 150, 170), ("South", 98, 110, 105, 130), ("West", 143, 120, 160, 175)]
for row in data:
    ws.append(row)

for r in range(2, ws.max_row + 1):                                  # formulas are just strings starting with =
    ws[f"F{r}"] = f"=SUM(B{r}:E{r})"
ws["A6"] = "Grand total"
ws["F6"] = "=SUM(F2:F4)"

header_font = Font(bold=True, color="FFFFFF")
for cell in ws[1]:                                                  # style the header row
    cell.font = header_font
    cell.fill = PatternFill("solid", fgColor="3776AB")
    cell.alignment = Alignment(horizontal="center")
for col in range(1, 7):
    ws.column_dimensions[get_column_letter(col)].width = 12
ws.freeze_panes = "A2"                                              # keep the header visible when scrolling

wb.save("sales_report.xlsx")
print("saved sales_report.xlsx")

saved sales_report.xlsx


In [6]:
wb = load_workbook("sales_report.xlsx")              # data_only=True would return cached formula RESULTS (needs Excel to have saved them)
ws = wb["Sales"]
print(wb.sheetnames, ws.dimensions)
print(ws["A2"].value, ws["B2"].value, ws["F2"].value)          # formulas come back as text
for row in ws.iter_rows(min_row=2, max_row=4, values_only=True):
    region, *quarters, _ = row
    print(f"{region:<6} total = {sum(quarters)}")

['Sales'] A1:F6
North 120 =SUM(B2:E2)
North  total = 575
South  total = 443
West   total = 598


> 💡 For analysis, **pandas** reads and writes Excel in one line: `pd.read_excel("file.xlsx")` / `df.to_excel("out.xlsx")`. For Google Sheets, use the `gspread` library.

## Part 3: The 📗 PDFs & Spreadsheets puzzle

**Task 1:** `find_the_link.csv` hides a Google Drive link. Each row holds one character of the link at the position equal to its row number (row 0 → column 0, row 1 → column 1, ...), i.e. on the **diagonal**.

In [7]:
import csv

with open("find_the_link.csv", encoding="utf-8", newline="") as f:
    data_lines = list(csv.reader(f))

link = "".join(row[i] for i, row in enumerate(data_lines))
print(link)

https://drive.google.com/open?id=1G6SEgg018UB4_4xsAJJ5TdzrhmXipr4Q


**Task 2:** The PDF linked there (`Find_the_Phone_Number.pdf`, included with the course) contains a phone number in the format `###.###.####`. Find it with a regular expression (Module 12):

In [8]:
import re

phone_reader = pypdf.PdfReader("Find_the_Phone_Number.pdf")
print("pages:", len(phone_reader.pages))
pattern = re.compile(r"\d{3}.\d{3}.\d{4}")
for n, page in enumerate(phone_reader.pages):
    match = pattern.search(page.extract_text())
    if match:
        print(f"Found on page {n}: {match.group()}")

pages: 17


Found on page 13: 505.503.4455


## 📝 Summary
- ✅ `pypdf` (not the deprecated PyPDF2): `PdfReader(path)`, `reader.pages`, `page.extract_text()`.
- `PdfWriter`: `add_page`, `append` (merge), `rotate`, `encrypt`, `write`.
- Generate PDFs with `fpdf2`/ReportLab.
- `openpyxl`: `Workbook`, `ws.append`, `ws["A1"]`, formulas, styles, `load_workbook`, `iter_rows(values_only=True)`.

## 🏋️ Practice
1. Count the words on each page of `Working_Business_Proposal.pdf`.
2. Create a PDF containing only the odd pages of `Working_Business_Proposal.pdf` in reverse order.
3. Write an Excel file with a multiplication table (1–10 × 1–10), bold headers, and a frozen first row/column.
4. Read it back and print the value at row 7, column 8.

<details><summary>Show solutions</summary>

```python
import pypdf
from openpyxl import Workbook, load_workbook
from openpyxl.styles import Font

r = pypdf.PdfReader("Working_Business_Proposal.pdf")
print([len(p.extract_text().split()) for p in r.pages])

w = pypdf.PdfWriter()
for page in reversed(r.pages[::2]):
    w.add_page(page)
w.write("odd_reversed.pdf")

wb = Workbook(); ws = wb.active
ws.append([""] + list(range(1, 11)))
for i in range(1, 11):
    ws.append([i] + [i * j for j in range(1, 11)])
for c in ws[1]: c.font = Font(bold=True)
for row in ws.iter_rows(min_col=1, max_col=1):
    row[0].font = Font(bold=True)
ws.freeze_panes = "B2"
wb.save("times.xlsx")

print(load_workbook("times.xlsx").active.cell(row=7, column=8).value)   # 6 × 7 = 42
```
</details>